# NanoGPT from Scratch
Following Andrej Karpathy's ["Let's build GPT"](https://www.youtube.com/watch?v=kCc8FmEb1nY) tutorial.

We will build a GPT — a character-level language model — completely from scratch, step by step.

---

## Step 1: Check our Environment

Before anything, we make sure PyTorch can see our GPU.  
The GPU will make training **100x faster** than the CPU for the matrix operations GPTs rely on.

In [35]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("No GPU found — training will be slow!")

PyTorch version: 2.11.0+cu128
CUDA available:  True
GPU: NVIDIA GeForce RTX 5060 Ti
VRAM: 17.1 GB


In [36]:
props = torch.cuda.get_device_properties(0)
print(f"Streaming Multiprocessors (SMs): {props.multi_processor_count}")
print(f"CUDA cores (SMs × 128):          {props.multi_processor_count * 128}")
print(f"VRAM:                             {props.total_memory / 1e9:.1f} GB")

Streaming Multiprocessors (SMs): 36
CUDA cores (SMs × 128):          4608
VRAM:                             17.1 GB


---
## Step 2: Download the Data — Tiny Shakespeare

We train on ~1MB of Shakespeare plays concatenated into one big text file.  
This is small enough to load entirely into memory, which keeps things simple.

The model's job: **given some characters, predict the next character**.  
After enough training, it will start generating Shakespeare-like text.

In [37]:
import urllib.request
import os

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "input.txt"

if not os.path.exists(DATA_PATH):
    print("Downloading tiny Shakespeare...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Done.")
else:
    print("Already downloaded.")

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Total characters in dataset: {len(text):,}")
print("\nFirst 100 characters:")
print(text[:100])

Already downloaded.
Total characters in dataset: 1,115,394

First 100 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


---
## Step 3: Build a Tokenizer

A **tokenizer** converts text (strings) into numbers, because neural networks only work with numbers.

We're using a **character-level tokenizer** — the simplest possible kind:
- Each unique character in the dataset gets a unique integer ID.
- `encode('hello')` → `[46, 43, 50, 50, 53]`
- `decode([46, 43, 50, 50, 53])` → `'hello'`

In [38]:
# Find every unique character in the dataset, sorted alphabetically
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size} unique characters")
print("All characters:", repr(''.join(chars)))

Vocabulary size: 65 unique characters
All characters: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"


In [39]:
# Build lookup tables: character <-> integer
stoi = { ch: i for i, ch in enumerate(chars) }  # string to int
itos = { i: ch for i, ch in enumerate(chars) }  # int to string

# define functions to encode text and decode integers

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

# Test it
test_str = "Hello, World!"
encoded = encode(test_str)
decoded = decode(encoded)

print(f"Original : {test_str}")
print(f"Encoded  : {encoded}")
print(f"Decoded  : {decoded}")

Original : Hello, World!
Encoded  : [20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Decoded  : Hello, World!


Note: stoi is a dictionary that maps a string/character to an integer. itos is a dictionary that maps an integer to a string/character. Here is what they look like

In [40]:
print(stoi)
print(itos)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i',

---
## Step 4: Encode the Dataset and Split into Train / Validation

We will encode the entire text as a tensor of integers, and then split it into training data (90%, we will train our gpt on this) and validation data (10%, will compare the model against this)

In [41]:
# Encode the full dataset into a tensor of integers
data = torch.tensor(encode(text), dtype=torch.long)

print(f"Data shape : {data.shape}")
print(f"Data dtype : {data.dtype}")
print(f"First 50 tokens: {data[:50]}")
print(f"Which decodes to: {repr(decode(data[:50].tolist()))}")

Data shape : torch.Size([1115394])
Data dtype : torch.int64
First 50 tokens: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])
Which decodes to: 'First Citizen:\nBefore we proceed any further, hear'


In [42]:
# Train / val split (90% / 10%)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f"\nTrain tokens: {len(train_data):,}")
print(f"Val tokens  : {len(val_data):,}")


Train tokens: 1,003,854
Val tokens  : 111,540


---
## Step 5: Data Loading — Batches and Context Windows

Instead of feeding the entire training dataset into the model, we will feed random chunks (called context windows or blocks).

Key concepts:
- **`block_size`** — how many characters the model sees at once (its "context window"). We use 8 for now.
- **`batch_size`** — how many independent chunks we process in parallel (one per GPU core, roughly). We will use 32 for now.

Given a string of tokens, the gpt model will predict the next token. For a chunk `[t1, t2, t3, t4, t5]`, the model actually trains on **4 examples simultaneously**:
- Input `[t1]` → predict `t2`
- Input `[t1, t2]` → predict `t3`
- Input `[t1, t2, t3]` → predict `t4`
- Input `[t1, t2, t3, t4]` → predict `t5`

In [43]:
# --- Hyperparameters ---
block_size = 8    # context window (how many characters the model looks back)
batch_size = 32   # how many sequences to process in parallel

# confirm that we are training on a GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")

Training on: cuda


In [44]:
torch.manual_seed(1337)  # makes random operations reproducible

# Show one example block to build intuition
x = train_data[:block_size]      # input:  first 8 tokens
y = train_data[1:block_size+1]   # target: next 8 tokens (shifted by 1)

print("\nExample input  x:", x.tolist(), "=", decode(x.tolist()))
print("Example target y:", y.tolist(), "=", decode(y.tolist()))
print()
for t in range(block_size):
    context = x[:t+1].tolist()
    target  = y[t].item()
    print(f"  context={decode(context)!r:20s}  →  next char={decode([target])!r}")


Example input  x: [18, 47, 56, 57, 58, 1, 15, 47] = First Ci
Example target y: [47, 56, 57, 58, 1, 15, 47, 58] = irst Cit

  context='F'                   →  next char='i'
  context='Fi'                  →  next char='r'
  context='Fir'                 →  next char='s'
  context='Firs'                →  next char='t'
  context='First'               →  next char=' '
  context='First '              →  next char='C'
  context='First C'             →  next char='i'
  context='First Ci'            →  next char='t'


Our goal is to randomly grab batch_size (=32) chunks of text from the dataset, each block_size (=8) characters long, packaged as tensors ready for the gpu. This sampling of x (input) and y (target) pairs needs to be done from both training and validation datasets

In [45]:
def get_batch(split):
    """Sample a random batch of (input, target) pairs from the dataset."""
    # select which dataset to sample from
    data = train_data if split == 'train' else val_data
    
    # Pick batch_size random starting positions
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # Stack into 2D tensors of shape (batch_size, block_size)
    x = torch.stack([data[i : i+block_size]   for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    
    # Move to GPU (or CPU if no GPU)
    x, y = x.to(device), y.to(device)
    return x, y

# Show what a batch looks like
xb, yb = get_batch('train')
print("Input batch  xb shape:", xb.shape)   # (32, 8)
print("Target batch yb shape:", yb.shape)   # (32, 8)
print("\nFirst 4 rows of xb (each row = one context window):")
for i in range(4):
    print(f"  row {i}: {repr(decode(xb[i].tolist()))}")

Input batch  xb shape: torch.Size([32, 8])
Target batch yb shape: torch.Size([32, 8])

First 4 rows of xb (each row = one context window):
  row 0: "Let's he"
  row 1: 'for that'
  row 2: 'nt that '
  row 3: 'MEO:\nI p'


**Tensor shapes** — we use `(B, T)` notation:
- `B` = Batch size (32)
- `T` = Time / sequence length (block_size = 8)

So `xb` is a `(32, 8)` matrix: 32 rows, each row is 8 character IDs.

---
## Step 6: The Bigram Language Model

We first start with the **simplest possible language model**: a Bigram (two-character) model. A bigram model predicts the next character **based only on the current character** — it ignores all history. 

It works using an **embedding table**: a matrix of shape `(vocab_size, vocab_size)` ie a 65 x 65 matrix in our case. In this matrix, any entry corresponding to X,Y can be explained as: given I am at character X, what is the likelihood that the next character is Y? Each row is a vector of scores called logits.    

In [46]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # Each token looks up its row in this table to get logits for the next token
        # create the embedding table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    # forward pass
    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensors of integers
        # for every integer in idx, fetch the corresponding row from the embeddings table
        logits = self.token_embedding_table(idx)  # (B, T, C)  where C = vocab_size

        if targets is None:
            # during generation, we can skip loss entirely
            loss = None
        else:
            B, T, C = logits.shape # get dimensions
            logits  = logits.view(B*T, C)   # reshape to (B*T, C) for cross_entropy
            targets = targets.view(B*T)     # reshape to (B*T,)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(idx)           # forward pass
            logits = logits[:, -1, :]          # take only the last time step: (B, C)
            probs  = F.softmax(logits, dim=-1) # convert logits to probabilities: (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)  # sample: (B, 1), sample one char from the prob distribution
            idx = torch.cat((idx, idx_next), dim=1)             # append: (B, T+1)
        return idx

# Instantiate the model and move it to GPU
model = BigramLanguageModel(vocab_size)
m = model.to(device)

print(f"Model parameters: {sum(p.numel() for p in m.parameters()):,}")
print(f"Model is on: {next(m.parameters()).device}")

Model parameters: 4,225
Model is on: cuda:0


---
## Step 7: Generate Text Before Training

Before training, the model's weights are **random**. We will now see what it will generates (most likely pure noise) to give us a baseline to compare against after training

In [47]:
# Start from a single newline token (index 0 in our vocabulary)
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# Run the generate loop 200 times, each time appending a new token
generated_ids = m.generate(context, max_new_tokens=200)

# Decode and print
generated_text = decode(generated_ids[0].tolist())
print("--- Generated text (before training) ---")
print(generated_text)

--- Generated text (before training) ---

jqfnxfRkRZ'Ndc.wf,ZWAO.zU,CbsK
bHiPWlkTBbzAuG:QaSKJO-33jMGF?KI3duM!bLVUYthgfjuDqca,xv.tbfF dXlAhcaAeuFwqcHwpRW$WHDyZaYzxzUYN&:
YV3&$-kpofCYdzvBbH&V!OW;KI!lPWcaeg irYeuEYnIciKOlSW;HmlAZWlGDKsSeUBqqW$nJ


---
## Step 8: Training the Bigram model

Training adjusts the model's weights (ie the embedding table) to minimise the **loss** (prediction error). Each iteration has the follwing steps
1. Sample a random batch from the training data
2. Forward pass: Run the data through the model, compute loss
3. Backward pass: compute gradients (ie how much to nudge each weight)
4. Update: Move weights in the direction that reduces loss

We also periodically evaluate on the **validation set** to track generalisation. At the end of the training step, we have a new embedding table.

In [48]:
# Training hyperparameters
max_iters    = 3000   # total training steps
eval_interval = 300   # evaluate every N steps
learning_rate = 1e-2  # how big a step the optimizer takes
eval_iters   = 200    # how many batches to average for loss estimation

@torch.no_grad()
def estimate_loss():
    """Estimate average loss on train and val sets (more reliable than a single batch)."""
    out = {}
    model.eval()   # switch to eval mode (disables dropout etc. — not used yet, but good habit)
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()  # switch back to training mode
    return out

# Optimizer — AdamW is the standard choice for transformers
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# the training loop
print("Starting training...")
for iter in range(max_iters):

    # Evaluate periodically
    if iter % eval_interval == 0: # if we hit the eval interval
        losses = estimate_loss()
        print(f"step {iter:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch from the training data
    xb, yb = get_batch('train')

    # Forward pass: get predictions and compute loss
    logits, loss = model(xb, yb)

    # Backward pass: compute gradients
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # Update weights
    optimizer.step()

print("\nTraining done!")

Starting training...
step    0: train loss 4.7676, val loss 4.7677
step  300: train loss 2.8381, val loss 2.8538
step  600: train loss 2.5531, val loss 2.5811
step  900: train loss 2.4967, val loss 2.5151
step 1200: train loss 2.4878, val loss 2.5086
step 1500: train loss 2.4677, val loss 2.4946
step 1800: train loss 2.4695, val loss 2.4961
step 2100: train loss 2.4707, val loss 2.4871
step 2400: train loss 2.4645, val loss 2.4892
step 2700: train loss 2.4736, val loss 2.4920

Training done!


---
## Step 9: Generate Text After Training

Now let's see what the model generates after learning from Shakespeare.

In [49]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(m.generate(context, max_new_tokens=500)[0].tolist())

print("--- Generated text (after training) ---")
print(generated_text)

--- Generated text (after training) ---

BE:
Wileranousel lind me l.
HAshe ce hiry:
Supr aisspllw y.
Hentofu n Boopetelaves
MPOFry wod mothakleo Windo whthCoribyo the m dourive we higend t so mower; te

AN ad nterupt f s ar igr t m:

Thin maleronth,
Mad
RD:

WISo myrangoube!
KENob&isarardsal thes ghesthinin couk ay aney Iry ts I fr y ce.
Jken pand, bemary.
Yof 'sour mend sora an hy t--pond bethe men.
Sand thowngulin s th llety ome.
I muco ffepyotssthecas l.
TAnEn s wethal wove.
se ed Pe bene oveveaimous?



AMPe cok hedin tie s inds he


The output is not coherent English — the bigram model only looks at one character at a time. But we notice notice:
- Real English-looking words start to appear.
- Character sequences that look like Shakespeare (thee, thou, wilt...)
- Reasonable punctuation patterns.

It's learned **which characters tend to follow which** — nothing more.


---
# Part 2: The Transformer

The bigram model is fundamentally limited: it only looks at the **immediately preceding character**.
To write anything coherent, a model needs to see much further back.

The transformer's key innovation is **self-attention**: a mechanism that lets every position in the sequence
look at every other position and decide what's relevant. In simpler terms, self attention allows a transformer to decide whether a letter at position 10 will have any impact on its prediction for the letter at position 20. 

We'll build up to it in 4 versions, each a small improvement on the last.

## Step 10a: Bag of words to aggregate Past Information?

Instead of a single number, imagine each token as a vector (or an **embedding**) of size `C`. At a given position `t`, we want to summarize the information provided by all positions from `0` to `t`.

**Version 1:** The simplest idea — just average all previous embeddings. This approach is called a "Bag of Words" - equal weight to everything, and order/relative positions don't matter.

In [52]:
# Version 1: naive loop — average all previous token embeddings
torch.manual_seed(1337)
B, T, C = 4, 8, 2   # batch=4, time=8, channels=2 (2-dimensional embeddings)
# start with random numbers between 0 and 1
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))   # xbow = 'bag of words'
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]              # all tokens from 0 to t, shape (t+1, C)
        xbow[b, t] = xprev.mean(dim=0)  # average across the time dimension

print('x[0] (original embeddings):')
print(x[0])
print()
print('xbow[0] (averaged embeddings):')
print(xbow[0])
print()
print('Check: xbow[0,0] == x[0,0] (only one token to average):', torch.allclose(xbow[0,0], x[0,0]))
print('Check: xbow[0,1] == mean of x[0,0:2]:', torch.allclose(xbow[0,1], x[0,:2].mean(0)))

x[0] (original embeddings):
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

xbow[0] (averaged embeddings):
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

Check: xbow[0,0] == x[0,0] (only one token to average): True
Check: xbow[0,1] == mean of x[0,0:2]: True


## Step 10b: The Matrix Multiplication Trick

The one-by-one loop is slow, because Python needs to loop over each of the B*T positions. The same averaging can be done in one operation using a **lower triangular matrix** (`tril`). The GPU can do this entire `(T,T) @ (B,T,C)` multiplication in one parallel operation — thousands of times faster than the Python loop.

In [56]:
# Version 2: matrix multiplication with tril
T = 8 # context window

# tril: a lower triangular matrix of ones
# row i has ones in columns 0..i, zeros in columns i+1..T-1
tril = torch.tril(torch.ones(T, T))
print('tril matrix:')
print(tril)

tril matrix:
tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])


In [60]:
# Divide each row by its sum so each row sums to 1 (averaging weights)
wei = tril / tril.sum(dim=1, keepdim=True)
print()
print('wei (normalised, each row sums to 1):')
print(wei)



wei (normalised, each row sums to 1):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


In [63]:
# Apply: wei @ x computes the weighted average for every position at once
xbow2 = wei @ x   # (T,T) @ (B,T,C) broadcasts to (B,T,C)
print('xbow (loop version):')
print(xbow[0])
print()
print('xbow2 (matrix version):')
print(xbow2[0])
print('xbow2 equals xbow (the loop version)?', torch.allclose(xbow, xbow2))

xbow (loop version):
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

xbow2 (matrix version):
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])
xbow2 equals xbow (the loop version)? False


Note: xbow and xbow2 are identical, but our code does not show them to be the same. This is an issue with floating point precision - the two matrices are extremely close but not bit-for-bit identical. We can see the maximum difference (absolute) below

In [64]:
diff = (xbow - xbow2).abs().max()
print('Max difference:', diff.item())

Max difference: 3.236345946788788e-08


## Step 10c: Softmax — Allowing Variable Weights

Equal averaging is not very expressive, since some past tokens may be more relevant than others. That's what softmax adds. Instead of equal weights, we start with zeros and use `softmax` to normalise. This opens the door to **learned weights**.

In [66]:
# Version 3: softmax version
# Start with zeros — becomes equal weights after softmax
# But crucially: we can REPLACE the zeros with learned values later

tril = torch.tril(torch.ones(T, T))
wei  = torch.zeros((T, T))

# Mask future positions with -inf so they become 0 after softmax
wei = wei.masked_fill(tril == 0, float('-inf'))
print('Before softmax (future positions = -inf):')
print(wei)

wei = F.softmax(wei, dim=-1)
print()
print('After softmax (future positions = 0):')
print(wei)

xbow3 = wei @ x
print()
print('xbow3 equals xbow?', torch.allclose(xbow, xbow3))
diff2 = (xbow - xbow3).abs().max()
print('Max difference:', diff2.item())

Before softmax (future positions = -inf):
tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

After softmax (future positions = 0):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        

Result is the same equal weights as before — but now the `zeros` can be replaced with **any learned values**. If we fill those zeros with data-dependent scores instead of zeros, the model can learn to pay more attention to some past tokens than others. That data-dependent filling is exactly what **keys and queries** do.

## Step 10d: Self-Attention — Queries, Keys, and Values

Each token produces three vectors:
- **Query** (`q`): "What am I looking for?"
- **Key** (`k`): "What do I contain / offer?"
- **Value** (`v`): "What do I actually send to others?"

The attention weight between positions `i` and `j` is how well `query[i]` matches `key[j]`. We compute this as a dot product: `q @ k.T`.

High dot product = strong match = pay more attention to that token.

In [67]:
# Version 4: Self-Attention with Queries, Keys, and Values
torch.manual_seed(1337)
B, T, C = 4, 8, 32   # larger channels now, use 32-dim embeddings
x = torch.randn(B, T, C) # random init

head_size = 16 #define dimensions of q,k,v matrices
key   = nn.Linear(C, head_size, bias=False)  # what do I offer?
query = nn.Linear(C, head_size, bias=False)  # what am I looking for?
value = nn.Linear(C, head_size, bias=False)  # what do I actually send?

k = key(x)    # (B, T, head_size)
q = query(x)  # (B, T, head_size)

# Attention scores: query dot-product with all keys
# Scale by 1/sqrt(head_size) to keep variance stable
wei = q @ k.transpose(-2, -1) * head_size**-0.5   # (B, T, T)

# Mask future positions (causal attention - can't look ahead)
tril = torch.tril(torch.ones(T, T))
wei  = wei.masked_fill(tril == 0, float('-inf'))
wei  = F.softmax(wei, dim=-1)  # (B, T, T)

# Weighted sum of values
v   = value(x)  # (B, T, head_size)
out = wei @ v   # (B, T, head_size)

print('Input x shape  :', x.shape)
print('Attention wei  :', wei.shape, ' <- for each token, weights over all past tokens')
print('Output shape   :', out.shape, ' <- richer representation for each position')
print()
print('Attention weights for position 0 (can only attend to itself):')
print(wei[0, 0])
print()
print('Attention weights for position 7 (can attend to all 8 positions):')
print(wei[0, -1])

Input x shape  : torch.Size([4, 8, 32])
Attention wei  : torch.Size([4, 8, 8])  <- for each token, weights over all past tokens
Output shape   : torch.Size([4, 8, 16])  <- richer representation for each position

Attention weights for position 0 (can only attend to itself):
tensor([1., 0., 0., 0., 0., 0., 0., 0.], grad_fn=<SelectBackward0>)

Attention weights for position 7 (can attend to all 8 positions):
tensor([0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553],
       grad_fn=<SelectBackward0>)


---
## Step 11: Self-Attention as a Class (one Head)

Now we package Version 4 into a reusable `Head` class.
Note: `n_embd`, `block_size`, and `dropout` will be defined in the hyperparameters cell below.

In [68]:
# Define hyperparameters first — Head and all subsequent classes use them
batch_size = 64 # number of independent sequences that can be processed in parallel in each training step
block_size = 256 # context window, how many tokens the model sees at once
n_embd  = 384 # embedding dimension, how many numbers represent a token
n_head  = 6 # number of attention heads running in parallel
n_layer = 6 # number of transformer blocks stacked on top of each other
dropout = 0.2 # randomly zero out this share of neurons at each step
learning_rate = 3e-4 # step size for optimizer
max_iters     = 5000 # total training steps
eval_interval = 500
eval_iters    = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'block_size: {block_size}, n_embd: {n_embd}, n_head: {n_head}, n_layer: {n_layer}')
print(f'Training on: {device}')

block_size: 256, n_embd: 384, n_head: 6, n_layer: 6
Training on: cuda


In [71]:
class Head(nn.Module):
    """One head of self-attention."""

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # tril is not a learned parameter — register_buffer keeps it on the right device
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # Attention scores
        wei = q @ k.transpose(-2, -1) * C**-0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Weighted aggregation of values
        v   = self.value(x)  # (B, T, head_size)
        out = wei @ v        # (B, T, head_size)
        return out

---
## Step 12: Multi-Head Attention

One attention head can only look for one kind of pattern at a time.
Running **multiple heads in parallel** lets the model attend to many different things simultaneously. For example: one head might learn to find subjects, another finds verbs, another tracks rhyme sounds.

Each head gets `n_embd / n_head` dimensions. Their outputs are concatenated to restore `n_embd` size.

Note that 6 is just a reasonable starting point. We can choose any value for n_head, but 
- n_head must divide evenly into n_embd
- More heads = smaller head_size per head. Each head has less "room" to work with.
- mMore heads = more parameters = slower training, with diminishing returns past a certain point

The original GPT-2 paper found that scaling up n_embd and n_layer mattered more than adding more heads. GPT-3 (175B parameters) uses 96 heads — but it also has n_embd = 12,288, so each head still gets 128 dimensions. As a rule of thumb: head_size should be at least 32–64 to give each head enough dimensions to be expressive. 

In [72]:
class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention running in parallel."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(n_embd, n_embd)  # mix the concatenated head outputs
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run all heads independently, concatenate along the last dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, n_embd)
        out = self.dropout(self.proj(out))
        return out

---
## Step 13: Feed-Forward Network

After attention, every position has gathered information from its context. But attention is just **aggregation** - it mixes information but doesn't deeply *process* it. The **feed-forward network** (FFN) is where each position independently thinks about what it just gathered. It's a simple two-layer MLP applied to every position identically.

Karpathy's framing: **attention is communication, feed-forward is computation.**

In [75]:
class FeedForward(nn.Module):
    """A simple MLP applied independently to each position."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # expand to 4x width
            nn.ReLU(),                        # non-linearity
            nn.Linear(4 * n_embd, n_embd),  # project back to n_embd
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

**Why both attention AND feed-forward?**
Attention communicates *between* positions (token A reads token B). After attention, each token's 384-number vector is a mixture of context. For example, the token 't' in "thou" might now contain:
- Information from 'h', 'o', 'u' via attention
- Some signal that this looks like an archaic word
- Some signal that a verb might follow

The FFN is where the model processes that context into something useful for prediction. Each of the 4 x n_embd act as detectors, which fire (or don't fire) based on the pattern they see in the 384 inputs. Then Linear(4 x n_embd → n_embd) looks at which neurons fired and writes that back into the token's representation, ready for the next layer. Each block's FFN builds on what the previous block's attention gathered.


---
## Step 14: Transformer Block

A **Block** combines multi-head attention + feed-forward, with two important additions:

### Residual Connections
Instead of `x = layer(x)`, we write `x = x + layer(x)`. The original input is added back to the output.
This gives gradients a direct path backward through all layers during training. Without this, deep networks fail to train — gradients shrink to zero before reaching early layers (the **vanishing gradient** problem).

### Layer Normalisation
`nn.LayerNorm(n_embd)` normalises each token's embedding to have mean 0 and variance 1, keeping activations in a stable range. Applied *before* each sub-layer (pre-norm variant, slightly different from the original paper but works better in practice).

In [76]:
class Block(nn.Module):
    """Transformer block: self-attention (communicate) then feed-forward (compute)."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head              # split embedding evenly across heads
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)          # layer norm before attention
        self.ln2  = nn.LayerNorm(n_embd)          # layer norm before feed-forward

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # attend, add residual
        x = x + self.ffwd(self.ln2(x))  # compute, add residual
        return x

The residual connections are why 96-layer models (like GPT-3) can be trained at all. Gradients flow back through the `+` without passing through layers, staying large enough to update early layers.

---
## Step 15: The Full GPT Model

Now we assemble everything into one class:

1. **Token embeddings** — same as before, now `n_embd`-dimensional (not `vocab_size`)
2. **Position embeddings** — NEW: a learnable vector for each position index 0 to `block_size-1`. Added to token embeddings so the model knows *where* each token is.
3. **N transformer Blocks** — stacked one after another
4. **Final LayerNorm** — applied after all blocks
5. **Linear head** — projects from `n_embd` back to `vocab_size` to get logits

**Why position embeddings?** Self-attention is order-agnostic — `q @ k.T` gives the same scores regardless of token order. Position embeddings inject order by giving each position a unique vector that gets added to its token embedding.

In [77]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)    # what token is it?
        self.position_embedding_table = nn.Embedding(block_size, n_embd)   # where is it?
        self.blocks  = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)          # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size) # project to vocab scores

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)                                # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embd)
        x = tok_emb + pos_emb    # (B, T, n_embd) — token identity + position
        x = self.blocks(x)       # (B, T, n_embd) — pass through all transformer blocks
        x = self.ln_f(x)         # (B, T, n_embd) — final normalisation
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits  = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]   # crop: position table only goes to block_size
            logits, loss = self(idx_cond)
            logits   = logits[:, -1, :]
            probs    = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx      = torch.cat((idx, idx_next), dim=1)
        return idx

# Instantiate and move to GPU
torch.manual_seed(1337)
gpt_model = GPTLanguageModel().to(device)

total_params = sum(p.numel() for p in gpt_model.parameters())
print(f'GPT parameters : {total_params:,}')
print(f'Bigram was     : 4,225 parameters')
print(f'That is        : {total_params // 4225:,}x more parameters')

GPT parameters : 10,788,929
Bigram was     : 4,225 parameters
That is        : 2,553x more parameters


---
## Step 16: Train the GPT

The training loop is identical to the bigram — only the model and hyperparameters change.

**Expected results:** train loss ~1.5, val loss ~1.6 after 5000 steps.  
Compare to bigram's ~2.5 — a massive improvement from being able to see 256 characters of context.

In [ ]:
# Re-define get_batch with updated block_size and batch_size
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x  = torch.stack([data[i : i+block_size]     for i in ix])
    y  = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    gpt_model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = gpt_model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    gpt_model.train()
    return out

optimizer = torch.optim.AdamW(gpt_model.parameters(), lr=learning_rate)

# Lists to track loss over time
train_losses = []
val_losses   = []
loss_steps   = []

print('Starting GPT training...')
for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        train_losses.append(losses['train'].item())
        val_losses.append(losses['val'].item())
        loss_steps.append(iter)
        print(f'step {iter:5d}: train loss {losses["train"]:.4f}, val loss {losses["val"]:.4f}')

    xb, yb = get_batch('train')
    logits, loss = gpt_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print('\nTraining done!')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(loss_steps, train_losses, label='train loss', color='blue')
plt.plot(loss_steps, val_losses,   label='val loss',   color='orange', linestyle='--')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('GPT Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f'Final train loss: {train_losses[-1]:.4f}')
print(f'Final val loss  : {val_losses[-1]:.4f}')
print(f'Gap (overfit indicator): {val_losses[-1] - train_losses[-1]:.4f}')

In [ ]:
# Generate text from the trained GPT
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(gpt_model.generate(context, max_new_tokens=500)[0].tolist())

print('--- Generated text (GPT after training) ---')
print(generated_text)

The output should now look dramatically more Shakespeare-like:
- Multi-word phrases that make grammatical sense
- Character names followed by colons (the dialogue format learned)
- Recognisable words, sentence structure, poetic rhythm

---
## What You Built

From scratch, in one notebook:

| Component | What it does |
|---|---|
| Token embeddings | Maps each character to an `n_embd`-dimensional vector |
| Position embeddings | Tells the model where in the sequence each token is |
| Self-attention (`Head`) | Every token attends to every past token, with learned weights |
| Multi-head attention | Multiple attention patterns running in parallel |
| Feed-forward | Each position independently processes what it gathered |
| Residual connections | Allow gradients to flow through deep networks |
| Layer norm | Keeps activations stable during training |
| Dropout | Prevents overfitting |

This is architecturally identical to GPT-2. The only differences between this and ChatGPT are scale (bigger model), data (vastly more), and compute (months on thousands of GPUs).